In [180]:
# ARTIFICIAL INTELLIGENCE GROUP PROJECT
#TITANIC SURVIVAL PREDICTION MOELING


In [181]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
%matplotlib inline

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_curve, roc_auc_score)

RANDOM_STATE = 42


In [182]:
# 1.0 LOAD THE DATASET

In [ ]:
#Pclass = Passenger class onboard the ship
#Sibsp = Siblings or spouses
#Parch = Parents or children aboard
#df = pd.read_csv(r"C:\Users\troyscoot\Downloads\New folder\Titanic-Dataset.csv")# THIS WAS FOR CALLING THE LOCAL CSV FILE
df = pd.read_csv("Titanic-Dataset.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()

In [ ]:
df.describe(include = "all")

In [186]:
# 2.0 EXPLORATORY DATA ANALYSIS

In [ ]:
# performing exploratory data analysis
#missing values

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

In [ ]:
#overall survival rate
survival_rate = df['Survived'].mean()
print(f"Overall survival rate: {survival_rate:.2%}")

plt.figure(figsize=(5,4))
sns.countplot(data=df, x='Survived', hue='Survived', palette='Set2', legend=False)
plt.title('Survival Count (0 = Died, 1 = Survived)')
plt.xlabel('Survived')
plt.ylabel('Number of Passengers')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.barplot(data=df, x='Sex', y='Survived', hue='Sex', palette='Set2', legend=False, ax=axes[0])
axes[0].set_title('Survival Rate by Sex')
axes[0].set_ylabel('Survival Rate')

sns.barplot(data=df, x='Pclass', y='Survived', hue='Pclass', palette='Set2', legend=False, ax=axes[1])
axes[1].set_title('Survival Rate by Passenger Class')
axes[1].set_ylabel('Survival Rate')

sns.barplot(data=df, x='Embarked', y='Survived', hue='Embarked', palette='Set2', legend=False, ax=axes[2])
axes[2].set_title('Survival Rate by Port of Embarkation')
axes[2].set_ylabel('Survival Rate')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data=df, x='Age', hue='Survived', multiple='stack', bins=30, palette='Set2')
plt.title('Age Distribution by Survival')
plt.show()

In [ ]:
# 3.0 DATA CLEANING

In [ ]:
#  performing data cleaning

df_clean = df.copy()

# Age: fill by median within Pclass + Sex group
df_clean['Age'] = df_clean.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# Embarked: fill with mode
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# Cabin -> binary indicator (had a recorded cabin or not)
df_clean['HasCabin'] = df_clean['Cabin'].notnull().astype(int)

# Drop columns we won't use as raw features
df_clean = df_clean.drop(columns=['PassengerId', 'Ticket', 'Cabin', 'Name'])

print("Remaining missing values:")
print(df_clean.isnull().sum())


In [ ]:
# 4.0 FEATURE ENGINEERING

In [ ]:


df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)

plt.figure(figsize=(7,4))
sns.barplot(data=df_clean, x='FamilySize', y='Survived', hue='FamilySize', palette='Set2', legend=False)
plt.title('Survival Rate by Family Size')
plt.show()


In [ ]:
# Encoding categorical variables
df_model = pd.get_dummies(df_clean, columns=['Sex', 'Embarked'], drop_first=True)
df_model.head()


In [ ]:
# 5.0 TRAINING / TEST SPLIT

In [ ]:

X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)




In [ ]:
# Scale numeric features (helps Logistic Regression converge and treat features fairly)
scaler = StandardScaler()
num_cols = ['Age', 'Fare', 'FamilySize', 'SibSp', 'Parch']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
# 6.0 MODEL TRAINING

In [ ]:
# Model 1: Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)  # tree-based models don't need scaling
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

In [ ]:
# 7.0 MODEL EVALUATION

In [ ]:
def evaluate(y_true, y_pred, y_proba, model_name):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

results = pd.DataFrame([
    evaluate(y_test, y_pred_lr, y_proba_lr, 'Logistic Regression'),
    evaluate(y_test, y_pred_rf, y_proba_rf, 'Random Forest')
]).set_index('Model').round(3)

results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (y_pred, name) in zip(axes, [(y_pred_lr, 'Logistic Regression'), (y_pred_rf, 'Random Forest')]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])
    ax.set_title(f'Confusion Matrix: {name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


In [ ]:
print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=['Died', 'Survived']))

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=['Died', 'Survived']))

In [ ]:
plt.figure(figsize=(6,5))

for y_proba, name in [(y_proba_lr, 'Logistic Regression'), (y_proba_rf, 'Random Forest')]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()


In [ ]:
# 5-fold cross-validation for a more robust accuracy estimate
cv_lr = cross_val_score(log_reg, X_train_scaled, y_train, cv=5, scoring='accuracy')
cv_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy')

print(f"Logistic Regression CV accuracy: {cv_lr.mean():.3f} (+/- {cv_lr.std():.3f})")
print(f"Random Forest CV accuracy:       {cv_rf.mean():.3f} (+/- {cv_rf.std():.3f})")

In [ ]:
# 8.0 FEATURE IMPORTANCE

In [ ]:

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, palette='viridis', legend=False)
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.show()

importances

In [ ]:
# Logistic Regression coefficients (direction matters here: positive = higher survival odds)
coef = pd.Series(log_reg.coef_[0], index=X.columns).sort_values()

plt.figure(figsize=(8,5))
colors = ['crimson' if c < 0 else 'seagreen' for c in coef.values]
plt.barh(coef.index, coef.values, color=colors)
plt.title('Logistic Regression Coefficients\n(Green = increases survival odds, Red = decreases it)')
plt.xlabel('Coefficient Value')
plt.show()


In [ ]:
# 8.0(b) STATISTICAL SGNIFICANCE OF KEY RELATIONSHIPS

In [ ]:

from scipy import stats

print("=== Chi-square tests: categorical features vs Survived ===")
for col in ['Sex', 'Pclass', 'Embarked']:
    ct = pd.crosstab(df_clean[col] if col in df_clean.columns else df[col], df['Survived'])
    chi2, pval, dof, exp = stats.chi2_contingency(ct)
    print(f"{col:10s}: chi2 = {chi2:8.2f}   p-value = {pval:.2e}")

print()
print("=== Independent t-tests: continuous features vs Survived ===")
for col in ['Age', 'Fare']:
    grp_survived = df[df.Survived == 1][col].dropna()
    grp_died = df[df.Survived == 0][col].dropna()
    t, p = stats.ttest_ind(grp_survived, grp_died, equal_var=False)
    print(f"{col:10s}: survived mean = {grp_survived.mean():6.2f}, died mean = {grp_died.mean():6.2f}, "
          f"t = {t:6.2f}, p-value = {p:.2e}")

In [ ]:
#9.0 CORRELATION ANALYSIS

In [ ]:

corr_cols = ['Survived','Pclass','Age','SibSp','Parch','Fare','FamilySize','IsAlone','HasCabin']
plt.figure(figsize=(8,6))
sns.heatmap(df_model[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (Numeric Features)')
plt.tight_layout()
plt.show()


In [ ]:
# 10.0 HYPER PARAMETER TUNING

In [ ]:

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Logistic Regression
lr_grid = {'C': [0.01, 0.1, 1, 10, 100]}
gs_lr = GridSearchCV(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), lr_grid, cv=cv, scoring='accuracy', n_jobs=-1)
gs_lr.fit(X_train_scaled, y_train)

# Random Forest
rf_grid = {'n_estimators':[100,200,300], 'max_depth':[4,6,8,None], 'min_samples_leaf':[1,2,4]}
gs_rf = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), rf_grid, cv=cv, scoring='accuracy', n_jobs=-1)
gs_rf.fit(X_train, y_train)

# Gradient Boosting (new third model)
gb_grid = {'n_estimators':[100,200], 'learning_rate':[0.01,0.05,0.1], 'max_depth':[2,3,4]}
gs_gb = GridSearchCV(GradientBoostingClassifier(random_state=RANDOM_STATE), gb_grid, cv=cv, scoring='accuracy', n_jobs=-1)
gs_gb.fit(X_train, y_train)

print("Best Logistic Regression params:", gs_lr.best_params_, " | CV accuracy:", round(gs_lr.best_score_, 4))
print("Best Random Forest params:      ", gs_rf.best_params_, " | CV accuracy:", round(gs_rf.best_score_, 4))
print("Best Gradient Boosting params:  ", gs_gb.best_params_, " | CV accuracy:", round(gs_gb.best_score_, 4))

best_lr, best_rf, best_gb = gs_lr.best_estimator_, gs_rf.best_estimator_, gs_gb.best_estimator_

In [ ]:
# 11.0 FINAL MODEL COMPARISONS(TUNED, 3 MODELS)

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

def full_eval(model, Xte, name):
    yp = model.predict(Xte)
    ypr = model.predict_proba(Xte)[:, 1]
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, yp),
        'Precision': precision_score(y_test, yp),
        'Recall': recall_score(y_test, yp),
        'F1-Score': f1_score(y_test, yp),
        'ROC-AUC': roc_auc_score(y_test, ypr),
        'PR-AUC': average_precision_score(y_test, ypr),
    }, yp, ypr

res_lr, yp_lr_t, ypr_lr_t = full_eval(best_lr, X_test_scaled, 'Logistic Regression (tuned)')
res_rf, yp_rf_t, ypr_rf_t = full_eval(best_rf, X_test, 'Random Forest (tuned)')
res_gb, yp_gb_t, ypr_gb_t = full_eval(best_gb, X_test, 'Gradient Boosting (tuned)')

final_results = pd.DataFrame([res_lr, res_rf, res_gb]).set_index('Model').round(4)
final_results


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (yp, name) in zip(axes, [(yp_lr_t,'Logistic Regression'), (yp_rf_t,'Random Forest'), (yp_gb_t,'Gradient Boosting')]):
    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=['Died','Survived'], yticklabels=['Died','Survived'])
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
for ypr, name in [(ypr_lr_t,'Logistic Regression'), (ypr_rf_t,'Random Forest'), (ypr_gb_t,'Gradient Boosting')]:
    fpr, tpr, _ = roc_curve(y_test, ypr)
    auc = roc_auc_score(y_test, ypr)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--', label='Random guess')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve Comparison (3 Models)'); plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(6,5))
for ypr, name in [(ypr_lr_t,'Logistic Regression'), (ypr_rf_t,'Random Forest'), (ypr_gb_t,'Gradient Boosting')]:
    prec, rec, _ = precision_recall_curve(y_test, ypr)
    ap = average_precision_score(y_test, ypr)
    plt.plot(rec, prec, label=f'{name} (AP={ap:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curve Comparison'); plt.legend()
plt.show()

In [ ]:
# 12.0 LEARNING CURVE TO CHOOSE THE BEST MODEL

In [ ]:

from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_rf, X_train, y_train, cv=cv, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8), random_state=RANDOM_STATE
)

plt.figure(figsize=(7,5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training score')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Validation score')
plt.xlabel('Training Set Size'); plt.ylabel('Accuracy'); plt.title('Learning Curve: Random Forest (tuned)'); plt.legend()
plt.show()

In [ ]:
# 13.0 ERROR ANALYSIS

In [ ]:
error_df = X_test.copy()
error_df['Actual'] = y_test
error_df['Predicted'] = yp_rf_t
error_df['Name'] = df.loc[X_test.index, 'Name']

misclassified = error_df[error_df.Actual != error_df.Predicted]
false_negatives = misclassified[misclassified.Actual == 1]   # predicted died, actually survived
false_positives = misclassified[misclassified.Actual == 0]   # predicted survived, actually died

print(f"Total misclassified: {len(misclassified)} / {len(error_df)} ({len(misclassified)/len(error_df):.1%})")
print(f"False negatives (missed survivors): {len(false_negatives)}")
print(f"False positives (missed deaths):    {len(false_positives)}")
print()
print("False negatives -- avg fare: {:.2f}, avg age: {:.1f}".format(false_negatives.Fare.mean(), false_negatives.Age.mean()))
print("False negatives -- Pclass distribution:")
print(false_negatives.Pclass.value_counts())
print()
print("False positives -- avg fare: {:.2f}, avg age: {:.1f}".format(false_positives.Fare.mean(), false_positives.Age.mean()))
print("False positives -- Pclass distribution:")
print(false_positives.Pclass.value_counts())